## Objetivo: analisar todas as válvula que de fato precisam elevar cota
---
**Etapas**
1. Extrair todas as medidas de válvulas que já foram encobertas no IFS
1. Tratar os dados para obter todas a válvulas que o ultimo report foi encoberta
2. Coletar todas as OS ativas e Histórico dessas válvula

In [1]:
#Importando bibliotecas
import pandas as pd


In [2]:
#abrindo o documento como um dataframe
df = pd.read_csv('medidas_todos_que_ja_foram_recapiadas.csv', sep=';', header=0, encoding='latin1')
#Filtrando apenas os dados consistentes
df = df[df['Tipo de Medição'] != 'Leitura Incorreta']
#Filtrando deixando apenas a ultima leitura
df = df.drop_duplicates(subset=['ID Objeto'], keep='last')
#Filtrando deixando apenas as recapeadas 'Valor Registrado' = 1
df = df[df['Valores Registrados'] > 0]


In [3]:
df.to_csv(
    'medidas_recapeadas_filtradas.csv',
    sep=';',
    index=False,
    encoding='latin1'
)

Nesse csv obtive essas TAGs de válvula
```CGR2-VE-03;CGR2-VE-04;CGR2-VE-26;RALG-022-01;RAMA-039-01;RATJ-01A-01;RCAC-015-01;RCM-03B-01;RCSL-010-01;RCSS-025-01;RCSS-025-02;RCXS-008-01;RDLN-059-01;RDUQ-060-01;RDZZ-030-01;REGE-013-01;RGLM-035-01;RHOG-015-01;RIGT-057-01;RJ23-30-01;RJD-001;RJMT-01A-01;RJNV-030-04;RLBD-024-01;RLVL-054-01;RMJA-038-01;RNAV-022-01;RNJB-060-01;RPET-041-01;RQJ-002;RRBA-029-01;RRNG-022-01;RRNG-022-02;RRP-027-01;RSGF-023-01;RSGF-023-02;RTBR-030-03;RTOE-029-03;RVDN-053-01;RVTB-042-02;RXVN-01A-01;TAJ-001;TAM-001;TBDT-002;TCE-002;TDIV-003;TEG-004;TEG-005;TEZ-003;TFM-002;TFZH-001;TJA-001;TJA-002;TML-002;TMSM-002;TMUR-002;TPM-003;TTBR-001;TTD-002```
---
Busquei todas essas válvulas no sistema com os seguintes objetivio
1. Obter o histórico de serviço nessas válvulas
2. Obter as solicitações de serviço para essas válvulas
---
O sistema me entregou essas duas planilhas CSV
* ```Historico_das_valvulas.csv```
* ```OSs_ativas_valvulas.csv```

In [4]:
#Abrindo OSs ativas
Historico_OS = pd.read_csv('Historico_das_valvulas.csv', sep=';', header=0, encoding='latin1')
OS_ativas = pd.read_csv('OSs_ativas_valvulas.csv', sep=';', header=0, encoding='latin1')

In [5]:
#Tabela de dados (Filtro: válvulas do df que não está em OS_ativas)
valvulas_sem_OS = df[~df['ID Objeto'].isin(OS_ativas['ID Objeto'])]


In [6]:
#Na tabela de histórico de serviço não preciso de todas as colunas apenas das colunas:
# Início Real
# Nº OS
# ID Objeto
# Diretriz
Historico_OS_tratada = Historico_OS[['Nº OS', 'Início Real', 'ID Objeto', 'Diretriz']]
Historico_OS_tratada

,Nº OS,Início Real,ID Objeto,Diretriz
0,21127,09/05/2024 17:10,CGR2-VE-03,Elevação de cota de caixa de válvula - CGR2-VE-03
1,21128,10/05/2024 08:11,CGR2-VE-04,Elevação de cota de caixa de válvula - CGR2-VE-04
2,25827,24/02/2025 10:23,RLBD-024-01,(AS 25) Elevação de cota da caixa de válvula -...
3,29095,19/08/2025 07:57,RSGF-023-01,Elevação de cota - RSGF-023-01
4,21125,01/04/2024 13:10,TDIV-003,Elevação de cota de caixa de válvula - TDIV-003
5,13625,25/01/2023 14:37,TJA-002,Elevação de cota de caixa de válvula - TJA-002...
6,7664,07/10/2019 09:23,TML-002,Acompanhar Empreiteira Elevação
7,21111,08/04/2024 15:13,TPM-003,Elevação de cota de caixa de válvula - TPM-003


In [7]:
#junta a tabela que vem das medidas do objeto com o historico do serviço
merge_dfANDHistorico_OS = df.merge(Historico_OS_tratada,
                                       on='ID Objeto',
                                       how='inner',
                                       suffixes=('_medidas', '_Historico'),
                                       indicator=True
)

In [8]:
#Válvulas que foi realizado serviço de elevação ou intalação de laje depois da inspeção
valvulas_OK = merge_dfANDHistorico_OS[merge_dfANDHistorico_OS['Data Registro'] > merge_dfANDHistorico_OS['Início Real']]
#Mudando o nome da coluna para deixar mais intiuitivo a leitura
valvulas_OK = valvulas_OK.rename(
    columns={'Início Real': 'Ultima elevação/instalação de laje',
             'Data Registro': 'Ultima inspeção'})
valvulas_OK


,Site do Objeto,ID Objeto,Descrição do Objeto,Cód Parâmetro,Valores Registrados,ID Ponto Teste,Descrição Ponto Teste,Unid Medida,Tipo de Medição,Descrição Parâmetros,Valor Medido,ID Medição,Nota de Medição,Ultima inspeção,Aviso,Recurso Seq.,Nº OS,Ultima elevação/instalação de laje,Diretriz,_merge
0,CGR,CGR2-VE-03,LINHA (PRÓX. BRITO & ROSSI) (Aço 10 '' /Tampa ...,ENCO,1.0,INSP,Inspeção da caixa,*,Leitura Registrada,Válvula encoberta/recapeada?,1.0,NaN,NaN,15/01/2026 15:31,Criado por ELVIS,NaN,21127,09/05/2024 17:10,Elevação de cota de caixa de válvula - CGR2-VE-03,both
3,CGR,RSGF-023-01,SALGADO FILHO - ESQ. AV. TIRADENTES (S/VENT 63MM),ENCO,1.0,INSP,Inspeção da caixa,*,Leitura Registrada,Válvula encoberta/recapeada?,1.0,NaN,NaN,20/12/2025 09:55,Criado por GUSTAVOG,NaN,29095,19/08/2025 07:57,Elevação de cota - RSGF-023-01,both
4,CGR,TDIV-003,R. da DIVISÃO - Esq. Tv. Baturité (160 mm/ Ta...,ENCO,1.0,INSP,Inspeção da caixa,*,Leitura Registrada,Válvula encoberta/recapeada?,1.0,NaN,NaN,23/03/2026 10:44,Criado por FERNANDO,NaN,21125,01/04/2024 13:10,Elevação de cota de caixa de válvula - TDIV-003,both
6,CGR,TML-002,MARQUÊS DE LAVRADIO - ESQ. TRÊS BARRAS (160 MM...,ENCO,1.0,INSP,Inspeção de caixa de válvula,*,Leitura Registrada,Válvula encoberta/recapeada?,1.0,NaN,NaN,19/01/2026 16:26,Criado por GABRIEL,NaN,7664,07/10/2019 09:23,Acompanhar Empreiteira Elevação,both
7,CGR,TPM-003,PAULO MACHADO - (160 mm/Tampa 60 cm),ENCO,1.0,INSP,Inspeção da caixa,*,Leitura Registrada,Válvula encoberta/recapeada?,1.0,NaN,NaN,20/01/2026 10:07,Criado por FERNANDO,NaN,21111,08/04/2024 15:13,Elevação de cota de caixa de válvula - TPM-003,both


---
### Resultados
1. Válvulas que precisa abrir OS
2. Válvulas que precisa atualizar os Status
3. Válvulas que estão pendentes para elevar/instalar laje que já tem OS e __não são bucha__
4. Válvulas que estão pendentes para elevar/instalar laje que já tem OS e __são bucha__

In [9]:
#1. Válvulas que precisa abrir OS
print('1. Válvulas que precisa abrir OS\n')
print(";".join(valvulas_sem_OS['ID Objeto'].tolist()))

1. Válvulas que precisa abrir OS

CGR2-VE-03;CGR2-VE-04;CGR2-VE-26;RALG-022-01;RAMA-039-01;RCM-03B-01;RCSL-010-01;RCSS-025-01;RCSS-025-02;RCXS-008-01;RDLN-059-01;RDUQ-060-01;RHOG-015-01;RIGT-057-01;RJ23-30-01;RJD-001;RLBD-024-01;RLVL-054-01;RMJA-038-01;RNAV-022-01;RRBA-029-01;RRP-027-01;RSGF-023-01;RSGF-023-02;RTBR-030-03;RTOE-029-03;RVDN-053-01;TCE-002;TDIV-003;TEG-004;TEG-005;TFM-002;TFZH-001;TTBR-001;TTD-002


In [10]:
#2. Válvulas que precisa atualizar os Status
print('2. Válvulas que precisa atualizar os Status\n')
print(';'.join(valvulas_OK['ID Objeto'].tolist()))

2. Válvulas que precisa atualizar os Status

CGR2-VE-03;RSGF-023-01;TDIV-003;TML-002;TPM-003


In [11]:
#3. Válvulas que estão pendentes para elevar/instalar laje que já tem OS e __não são bucha__
print('3. Válvulas que estão pendentes para elevar/instalar laje que já tem OS enão são bucha\n')
# As valvula TMUR-002, TBDT-002, TAJ-001, TPM-003 são bucha, já tentou executar o serviço mas posivelmente não há tampa ou precisa realizar grande escavação
print(';'.join(OS_ativas['ID Objeto'].tolist()).replace('TMUR-002', "").replace(
    'TBDT-002', '').replace(
    'TAJ-001', '').replace(
    'TPM-003', '').replace(
    'RSGF-023-02', ''))

3. Válvulas que estão pendentes para elevar/instalar laje que já tem OS enão são bucha

RRNG-022-01;TJA-001;REGE-013-01;;RNJB-060-01;;;;TEZ-003;RQJ-002;TMSM-002;TAM-001;RJNV-030-04;RRNG-022-02;RGLM-035-01;TML-002;RPET-041-01;RVTB-042-02;TJA-002;RJMT-01A-01;RXVN-01A-01;RCAC-015-01;RDZZ-030-01;RATJ-01A-01


In [12]:
#Sabe-se especificamente como o conhecimento do técnico que essas válvulas são problematicas, não sendo possivel filtra-las
print('4. Válvulas que são bucha \n')
print('TMUR-002;TBDT-002;TAJ-001;TPM-003;RSGF-023-02')

4. Válvulas que são bucha 

TMUR-002;TBDT-002;TAJ-001;TPM-003;RSGF-023-02
